# **Preparación del Notebook:**
- Para poder ejecutarse es necesario cargar el archivo df_unificado en /content/ mediante el menú lateral de la izquierda. Este archivo se encuentra en la carpeta del proyecto `datos_procesados`

In [ ]:
import os
from pathlib import Path                                                  # Importación del módulo de gestión de rutas
import pandas as pd                                                       # Framework principal para la manipulación y análisis de datos tabulares

ruta_local_colab = Path('/content/df_unificado.csv')
PATH_META_INFO = ruta_local_colab

df_unificado = None

if PATH_META_INFO.exists():
    df_unificado = pd.read_csv(PATH_META_INFO)
    print(f"✅ Datos cargados correctamente: {len(df_unificado)} registros")
else:
    print(
        f"\n❌ ERROR: No se encuentra 'df_unificado.csv' en la ruta esperada:\n"
        f"   -> '{PATH_META_INFO.resolve()}'\n\n"
        "⚠️ SOLUCIÓN RÁPIDA:\n"
        "   1. Sube manualmente 'df_unificado.csv' a la carpeta /content de este entorno de Colab\n"
        "      (panel izquierdo -> icono de carpeta -> arrastra el archivo o usa 'Subir').\n"
        "   2. Vuelve a ejecutar esta celda una vez subido el archivo."
    )
    raise SystemExit("Ejecución detenida: falta 'df_unificado.csv'.")

✅ Datos cargados correctamente: 6335 registros


In [ ]:
# 1. Instalamos dependencias y los módulos necesarios de LangChain
!pip install -q faiss-cpu langchain-community langchain langchain-core langchain-openai

# NOTA: pip puede mostrar un aviso de "dependency conflict" con el paquete interno google-colab
# (versión de 'requests'). Es un aviso, no un error: la instalación se completa igualmente.

# 🔎 Solución de IA Generativa



## 1- Introducción y contexto del problema

En el ecosistema digital actual, la desinformación representa un desafío crítico para la confianza del usuario. En el marco del desarrollo de **PontiApp**, nos enfrentamos a la necesidad operativa de gestionar un dataset compuesto por **6.335 registros de noticias**.

La moderación de este volumen de información mediante revisión exclusivamente humana resulta ineficiente, costosa y poco escalable. Además, el simple hecho de clasificar una noticia como `REAL` o `FAKE` mediante algoritmos tradicionales genera una "caja negra" que no ayuda al revisor humano a entender el porqué de la decisión.

**Objetivo del Proyecto:**
El presente proyecto tiene como objetivo diseñar e implementar un sistema de **Inteligencia Artificial Generativa y Búsqueda Semántica** que no solo clasifique la desinformación, sino que actúe como un "Asistente de Moderación". Buscamos dotar al sistema de **interpretabilidad**, generando justificaciones en lenguaje natural y permitiendo consultas semánticas profundas sobre el histórico de datos.

## 2- Exploración de soluciones y selección de propuesta

Para abordar el reto metodológico, hemos evaluado distintas arquitecturas de IA aplicadas a texto, descartando enfoques superficiales para centrarnos en tres propuestas avanzadas:

1. **Propuesta A: Sistema de Scoring y Justificación (Prompt Engineering Dinámico).**
   Consiste en un pipeline de auditoría que filtra noticias por reglas de negocio (ej. longitud manejable), utiliza un LLM para evaluar la "evidencia de falsedad" otorgando un *Score* del 1 al 10, y finalmente redacta un informe técnico explicando qué patrones manipulativos (clickbait, falta de fuentes, sesgos...) justifican la alerta.

2. **Propuesta B: Motor de Búsqueda Semántica (RAG con FAISS).**
   Implementación de una arquitectura *Retrieval-Augmented Generation*. Mediante la vectorización de las 6.335 noticias usando *OpenAI Embeddings*, este sistema permite al usuario "preguntar" a la base de datos. Supera las limitaciones de las búsquedas por palabras clave al entender el contexto matemático y semántico de los textos.

3. **Propuesta C: Categorización temática dinámica.**
  Consiste en utilizar un Modelo de Lenguaje para procesar el texto de las 6.335 noticias y asignar automáticamente una macro-categoría (Política, Salud, Economía, etc.) a cada registro. Este proceso enriquece semánticamente el dataset, estructurando la información para facilitar la identificación de tendencias y su posterior integración en dashboards de Business Intelligence.


Tras la fase de experimentación, la decisión arquitectónica ha sido **descartar la tercera opción para probar una solución de IA más elaborada y así integrar las dos primeras como un ecosistema dual** para PontiApp:

* Seleccionamos el **Sistema de Scoring (Propuesta A)** como herramienta de *Moderación Activa*. Es ideal para el flujo de entrada diario, donde cada noticia nueva es auditada automáticamente antes de su publicación.
* Seleccionamos el **Motor RAG (Propuesta B)** como herramienta de *Auditoría e Investigación*. Es indispensable para que los administradores analicen tendencias históricas y recuperen contexto sobre narrativas de desinformación previas.

A continuación, se presenta la ejecución y el análisis de ambas soluciones, optimizadas para su correcta visualización y rendimiento en el entorno.





##  3- 🤖 Ejecución de soluciones

### 3.1. Generador de Alertas y Justificaciones

####  3.1.1. Desarrollo de la solución de IA generativa

⚠️ **Nota para uso en Colab:** si el Run All completo falla la primera vez pidiendo la API Key, ejecuta esta celda sola, concede el acceso a Secretos cuando te lo pida Colab, y repite Run All.

In [ ]:
import os
from google.colab import userdata
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from getpass import getpass

# --- Lectura de la API key desde los Secretos de Colab ---
# Cada persona que ejecute el notebook usa SU PROPIA key, guardada en el panel
# de Secretos (icono de llave 🔑, a la izquierda), con el nombre OPENAI_API_KEY.
# Así nadie comparte su clave: el notebook lee la del usuario que lo ejecuta.

API_KEY = userdata.get('OPENAI_API_KEY')
print("✅ API Key cargada desde los Secretos de Colab.")

# Inicializamos el modelo (usamos gpt-4o-mini por ser rápido, eficiente y muy barato para pruebas)
llm = ChatOpenAI(temperature=0.5, openai_api_key=API_KEY, model_name="gpt-4o-mini")

# Creamos el Prompt Template (Las instrucciones para la IA)
template = """
Eres un asistente experto en detección de desinformación y detección de fake news para la plataforma PontiApp.
Nuestro algoritmo matemático ha marcado la siguiente noticia como 'Sospechosa / Fake'.

REGLAS PARA EL ANÁLISIS:
- Evita usar siempre las mismas frases introductorias. No siempre iniciar con "La noticia presenta...". Varia completamente la estructura de cada frase.
- Ve directo al grano. Identifica el patrón sospechoso inmediatamente.
- Analiza aspectos específicos: busca lenguaje sensacionalista, clickbait, subjetividad, falta de contraste, apelación a la emoción, uso de emojis en noticias serias, spam de links, videos o fotos, noticias repetidas o posibles errores gramaticales.
- Si una noticia es técnica, enfócate en la falta de fuentes. Si es política, enfócate en el sesgo.
- Sé directo y variado en el estilo. Máximo 3 líneas.

Título de la noticia: {titulo}
Texto de la noticia: {texto}

Alerta para el moderador:
"""

prompt = PromptTemplate(
    input_variables=["titulo", "texto"],
    template=template
)

# Creamos la cadena (Chain) que une el prompt con el modelo
chain = prompt | llm

✅ API Key cargada desde los Secretos de Colab.


##### Solución

Matriz de rendimiento

In [ ]:
# Para la prueba, filtramos solo 5 noticias aleatorias clasificadas como Fake para no gastar saldo innecesario
# Semilla fija (SEED=42, igual que el resto del proyecto) para que sea reproducible sin alterar resultados
df_fakes = df_unificado[df_unificado['fake'] == 1].sample(5, random_state=42).copy()
df_fakes['justificacion_ia'] = ""

# Creamos la columna vacía donde guardaremos la respuesta de la IA
print("🤖 Iniciando análisis generativo de alertas...\n")

for index, row in df_fakes.iterrows():

    titulo_noticia = row['titulo']
    texto_noticia = row['texto']
    id_noticia = row['t_id']

    try:
        # Invocamos a la IA pasándole los datos de cada fila
        respuesta = chain.invoke({"titulo": titulo_noticia, "texto": texto_noticia})

        # Guardamos la respuesta generada en nuestra nueva columna
        df_fakes.at[index, 'justificacion_ia'] = respuesta.content
        print(f"✅ Noticia ID {id_noticia} analizada correctamente.")

    except Exception as e:
        print(f"❌ Error en la noticia ID {id_noticia}: {e}")

# Ajustamos la vista de Pandas para poder leer el texto completo y mostramos el resultado
pd.set_option('display.max_colwidth', None)
display(df_fakes[['t_id', 'titulo', 'justificacion_ia']].head())

🤖 Iniciando análisis generativo de alertas...

✅ Noticia ID 9624 analizada correctamente.
✅ Noticia ID 5873 analizada correctamente.
✅ Noticia ID 5566 analizada correctamente.
✅ Noticia ID 5886 analizada correctamente.
✅ Noticia ID 7865 analizada correctamente.


,t_id,titulo,justificacion_ia
5789,9624,"WSJ Report Not About Black Rapists At Baylor, But Strictly About The White Guys Who Covered Up For Them","La noticia carece de credibilidad debido a su lenguaje sensacionalista y a la evidente falta de fuentes verificables. Además, el enfoque en la narrativa de victimización y la apelación emocional son características típicas de contenido sesgado. La promoción de un producto específico, como VDARE QUARTERLY, sugiere un interés más en la venta que en la información objetiva, lo que refuerza su naturaleza sospechosa."
3517,5873,Time: Investigating Hillary is an Attack on All Women,"El artículo presenta un lenguaje altamente emocional y sensacionalista, sugiriendo que la investigación a Hillary Clinton es un ataque a todas las mujeres, lo que carece de un análisis objetivo. La falta de fuentes y la apelación a la victimización generan dudas sobre su veracidad. Además, el tono condescendiente y los ataques personales hacia el autor y la figura de Clinton refuerzan un sesgo evidente en la narrativa."
3341,5566,Here Is Why the Trump Administration Will Be a Force for Peace - James P. Pinkerton,"El artículo presenta un sesgo evidente al defender de manera entusiasta la figura de Trump, utilizando un lenguaje que podría considerarse sensacionalista y subjetivo. Además, carece de fuentes contrastadas y se apoya en afirmaciones categóricas sin evidencia sólida que las respalde. La estructura del texto, junto con la llamada a la acción para donar al final, sugiere un enfoque más propagandístico que informativo."
3525,5886,Human Rights group alarmed at extra-judicial killings of MQM workers' by para-military force in Pakistan,"El contenido presenta un lenguaje altamente emotivo y sensacionalista, lo que puede desviar la atención de los hechos. La falta de fuentes verificables y la dependencia de declaraciones de un solo grupo (MQM) generan un sesgo evidente. Además, la repetición de acusaciones graves sin evidencia sólida refuerza la sospecha de desinformación."
4729,7865,Kurds decide to get on US nerves,"El artículo presenta un enfoque sensacionalista al sugerir que los kurdos están ""provocando"" a Estados Unidos sin proporcionar evidencia concreta. La falta de fuentes verificables, junto con la mezcla de comentarios de expertos sin un contexto claro, genera dudas sobre la veracidad del contenido. Además, el lenguaje utilizado es subjetivo y se percibe una clara inclinación política, lo que refuerza la sospecha de desinformación."


In [ ]:
import time

# Lista para guardar los tiempos de cada ejecución
tiempos_ejecucion = []

print("🤖 Iniciando análisis con medición de rendimiento...\n")

for index, row in df_fakes.iterrows():
    inicio = time.time() # Marcamos el tiempo de inicio

    try:
        respuesta = chain.invoke({"titulo": row['titulo'], "texto": row['texto']})
        df_fakes.at[index, 'justificacion_ia'] = respuesta.content

        fin = time.time() # Marcamos el tiempo al finalizar
        tiempos_ejecucion.append(fin - inicio) # Calculamos la diferencia

        print(f"✅ Noticia ID {row['t_id']} analizada en {fin - inicio:.2f} segundos.")
    except Exception as e:
        print(f"❌ Error: {e}")

# Cálculo final
promedio = sum(tiempos_ejecucion) / len(tiempos_ejecucion)
print(f"\n--- RENDIMIENTO ---")
print(f"Tiempo medio de procesamiento: {promedio:.2f} segundos por noticia.")

🤖 Iniciando análisis con medición de rendimiento...

✅ Noticia ID 9624 analizada en 2.45 segundos.
✅ Noticia ID 5873 analizada en 1.93 segundos.
✅ Noticia ID 5566 analizada en 1.84 segundos.
✅ Noticia ID 5886 analizada en 2.07 segundos.
✅ Noticia ID 7865 analizada en 2.11 segundos.

--- RENDIMIENTO ---
Tiempo medio de procesamiento: 2.08 segundos por noticia.


#### 3.1.2. Conclusiones y Valor Aportado



La implementación de este pipeline demuestra el enorme potencial de integrar Inteligencia Artificial Generativa para enriquecer los flujos de trabajo tradicionales. Hemos logrado superar la barrera de la "caja negra" algorítmica: el sistema no se limita a emitir un veredicto binario, sino que genera una justificación interpretable y estructurada.

* **Impacto Directo en Operaciones (PontiApp):** Esta solución reduce drásticamente el coste cognitivo y el tiempo invertido por los moderadores humanos. En lugar de analizar un texto completo desde cero, el analista recibe una alerta instantánea focalizada en los patrones de lenguaje engañoso detectados (ej. uso de *clickbait*, tono alarmista o falta de contraste en las fuentes).
* **Escalabilidad y Control de Costes:** El diseño arquitectónico de este modelo se basa en la eficiencia. Al aplicar el procesamiento de la API (OpenAI) únicamente sobre un subconjunto de registros pre-filtrados o con alto riesgo de falsedad, optimizamos radicalmente el consumo de tokens y minimizamos la latencia. Esto garantiza que el sistema sea económicamente viable en un entorno de producción real que deba procesar miles de noticias diarias.

Candidatos para la presentación:

In [ ]:
import textwrap

# 1. Filtramos las noticias cortas (150-300 caracteres)
df_candidatos = df_unificado[
    (df_unificado['fake'] == 1) &
    (df_unificado['texto'].str.len() >= 150) &
    (df_unificado['texto'].str.len() <= 300)
].copy()

# Tomamos una muestra de 10 para auditar (para que sea rápido y gaste pocos tokens)
df_candidatos = df_candidatos.sample(10, random_state=42)

# 2. DEFINIMOS EL SCORING (El "auditor" que busca las más exageradas)
template_scoring = """
Analiza la siguiente noticia y asígnale una puntuación de 'Evidencia de Fake News' del 1 al 10.
1 = Es muy sutil o poco evidente.
10 = Es un ejemplo perfecto de noticia falsa, con lenguaje alarmista, sin fuentes y clickbait claro.
Responde ÚNICAMENTE con el número.

Título: {titulo}
Texto: {texto}
Puntuación:
"""
prompt_scoring = PromptTemplate(input_variables=["titulo", "texto"], template=template_scoring)
chain_scoring = prompt_scoring | llm

print("📊 Auditando 10 candidatos para encontrar los 3 mejores ejemplos...\n")

# Calculamos el score de esos 10 candidatos
try:
    df_candidatos['score'] = df_candidatos.apply(
        lambda row: int(chain_scoring.invoke({"titulo": row['titulo'], "texto": row['texto']}).content.strip()),
        axis=1
    )
except Exception as e:
    print(f"Error en el scoring: {e}")
    # Si falla al convertir a número, ponemos un 0 por defecto
    df_candidatos['score'] = 0

# 3. Nos quedamos con el TOP 3 (las que tienen mayor nota)
df_top_3 = df_candidatos.sort_values(by='score', ascending=False).head(3)

print("🏆 CASOS DE ÉXITO PARA EL TRIBUNAL (FORMATO INFORME)\n")

ancho = 155

# 4. Generamos la justificación de la IA y lo imprimimos con textwrap
for index, row in df_top_3.iterrows():
    # Usamos tu 'chain' original (la de las alertas) para redactar el porqué
    justificacion = chain.invoke({"titulo": row['titulo'], "texto": row['texto']}).content

    print("=" * ancho)
    print(f"ID: {row['t_id']}  |  SCORE DE FALSEDAD: {row['score']}/10")
    print("-" * ancho)

    print("TÍTULO:")
    print(textwrap.fill(row['titulo'], ancho))
    print("-" * ancho)

    print("TEXTO ORIGINAL:")
    print(textwrap.fill(row['texto'], ancho))
    print("-" * ancho)

    print("JUSTIFICACIÓN IA:")
    print(textwrap.fill(justificacion, ancho))

    print("=" * ancho + "\n")

📊 Auditando 10 candidatos para encontrar los 3 mejores ejemplos...

🏆 CASOS DE ÉXITO PARA EL TRIBUNAL (FORMATO INFORME)

ID: 6719  |  SCORE DE FALSEDAD: 8/10
-----------------------------------------------------------------------------------------------------------------------------------------------------------
TÍTULO:
Amazing baby! 4-year-old Bella from Moscow easily speaks 7 languages
-----------------------------------------------------------------------------------------------------------------------------------------------------------
TEXTO ORIGINAL:
Notify me of follow-up comments by email. Notify me of new posts by email. Security Question: What is 12 + 15 ? Please leave these two fields as-is:
IMPORTANT! To be able to proceed, you need to solve the following simple math (so we know that you are a human) :-) Doom and Bloom
-----------------------------------------------------------------------------------------------------------------------------------------------------------
J

### 3.2. Asistente RAG

#### 3.2.1. Desarrollo de la solución de IA generativa

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document

# Cada noticia se indexa como UN único documento completo, sin fragmentarla en chunks:
# su longitud cabe dentro de la ventana de embeddings y es una unidad semántica coherente por sí sola.

print("🔄 Convirtiendo el dataset a formato Document...")
# 2. Convertimos las noticias
documentos = [
    Document(
        page_content=str(row['texto']),
        metadata={"titulo": str(row['titulo']), "t_id": str(row['t_id'])}
    )
    for _, row in df_unificado.iterrows()
]
print(f"✅ {len(documentos)} documentos preparados.")

🔄 Convirtiendo el dataset a formato Document...
✅ 6335 documentos preparados.


In [ ]:
import time
from openai import RateLimitError

print("🧠 Creando la base de datos vectorial con FAISS por bloques...")

# 1. Inicializamos las embeddings limpias (sin parámetros raros que rompan)
embeddings = OpenAIEmbeddings(
    openai_api_key=API_KEY,
    model="text-embedding-3-small"
)

# 2. Procesamos el dataset en tandas de 100 documentos
lote_tamano = 250
pausa_entre_lotes = 10
max_reintentos = 5
vectorstore = None

def procesar_lote_con_reintentos(lote, es_primero):
    global vectorstore
    for intento in range(1, max_reintentos + 1):
        try:
            if es_primero:
                vectorstore = FAISS.from_documents(lote, embeddings)
            else:
                vectorstore.add_documents(lote)
            return
        except RateLimitError:
            espera = 15 * intento
            print(f"   ⏳ Límite de tokens alcanzado (intento {intento}/{max_reintentos}). Esperando {espera}s...")
            time.sleep(espera)
    raise RuntimeError("No se pudo procesar el lote tras varios reintentos. Revisa tu cuota de OpenAI.")

for i in range(0, len(documentos), lote_tamano):
    lote_actual = documentos[i:i + lote_tamano]
    print(f"📦 Procesando lote {i // lote_tamano + 1} (Documentos del {i} al {i + len(lote_actual)})...")
    procesar_lote_con_reintentos(lote_actual, es_primero=(vectorstore is None))
    time.sleep(pausa_entre_lotes)

print("✅ Base de datos vectorial FAISS creada con éxito y lista para consultar.")

# 3. Guardamos en local para asegurar el trabajo
vectorstore.save_local("faiss_index_noticias")
print("💾 Índice guardado localmente en 'faiss_index_noticias'")

🧠 Creando la base de datos vectorial con FAISS por bloques...
📦 Procesando lote 1 (Documentos del 0 al 250)...
📦 Procesando lote 2 (Documentos del 250 al 500)...
📦 Procesando lote 3 (Documentos del 500 al 750)...
📦 Procesando lote 4 (Documentos del 750 al 1000)...
📦 Procesando lote 5 (Documentos del 1000 al 1250)...
📦 Procesando lote 6 (Documentos del 1250 al 1500)...
📦 Procesando lote 7 (Documentos del 1500 al 1750)...
📦 Procesando lote 8 (Documentos del 1750 al 2000)...
📦 Procesando lote 9 (Documentos del 2000 al 2250)...
📦 Procesando lote 10 (Documentos del 2250 al 2500)...
📦 Procesando lote 11 (Documentos del 2500 al 2750)...
📦 Procesando lote 12 (Documentos del 2750 al 3000)...
📦 Procesando lote 13 (Documentos del 3000 al 3250)...
📦 Procesando lote 14 (Documentos del 3250 al 3500)...
📦 Procesando lote 15 (Documentos del 3500 al 3750)...
📦 Procesando lote 16 (Documentos del 3750 al 4000)...
📦 Procesando lote 17 (Documentos del 4000 al 4250)...
📦 Procesando lote 18 (Documentos del 4

In [ ]:
import textwrap

def consultar_noticias(pregunta, num_noticias):
    """Busca las noticias más relevantes limitadas por 'k'"""
    k = max(1, min(int(num_noticias), 10))
    docs = vectorstore.similarity_search(pregunta, k=k)
    return docs

def mostrar_resultados(query, resultados):
    """Formatea y muestra los resultados por pantalla, limitando a 2500 caracteres"""
    print(f"\n" + "="*155)
    print(f" RESULTADOS PARA: '{query}'")
    print("="*155 + "\n")

    for i, doc in enumerate(resultados):
        print(f"--- RESULTADO {i+1} ---")
        print(f"ID: {doc.metadata['t_id']}")

        print("TÍTULO:")
        print(textwrap.fill(doc.metadata['titulo'], 155))
        print("-" * 155)

        # --- REGLA DE NEGOCIO: Límite de 2500 caracteres ---
        texto_noticia = doc.page_content
        if len(texto_noticia) > 2500:
            texto_noticia = texto_noticia[:2495] + " [...]"

        print(textwrap.fill(texto_noticia, 155))
        print("-" * 155)

        # Literal vs Semántico
        query_palabras = query.lower().split()
        texto_lower = doc.page_content.lower()
        palabras_clave = [p for p in query_palabras if len(p) > 3]
        coincidencias = [p for p in palabras_clave if p in texto_lower]

        if coincidencias:
            print(f"🔍 TIPO DE RESULTADO: Literal (Se encontraron las palabras exactas: {coincidencias})")
        else:
            print(f"🧠 TIPO DE RESULTADO: Semántico (FAISS lo relacionó por contexto)")

        print("="*155 + "\n")

##### Solución

In [ ]:
# =====================================================================
# BÚSQUEDA DIRECTA (Formato Normal LangChain)
# =====================================================================
print("🚀 EJECUTANDO BÚSQUEDA DIRECTA POR CÓDIGO...")
query_directa = "fraudes electorales"
resultados_directos = consultar_noticias(query_directa, 3)
mostrar_resultados(query_directa, resultados_directos)

🚀 EJECUTANDO BÚSQUEDA DIRECTA POR CÓDIGO...

 RESULTADOS PARA: 'fraudes electorales'

--- RESULTADO 1 ---
ID: 7980
TÍTULO:
More Reports Of Votes Flipping From Trump To Clinton In Texas ….election Officials Dismiss Concerns… An Ongoing Cbs4 Voter Fraud Investigation Finds People
Voting Twice
-----------------------------------------------------------------------------------------------------------------------------------------------------------
More Reports Of Votes Flipping From Trump To Clinton In Texas ….election Officials Dismiss Concerns… An Ongoing Cbs4 Voter Fraud Investigation Finds People
Voting Twice by IWB · October 27, 2016 Tweet Following our story yesterday, there have been more reports of early voters in Texas seeing their ballots
flipped from Donald Trump to Hillary Clinton. Voters in Arlington and Amarillo complained that when they highlighted the box to select Trump/Pence, it
switched to Clinton/Kaine. Now numerous other Texans have gone public on social media to repor

Si queremos hacerlo interactivo:

In [ ]:
# =====================================================================
# BÚSQUEDA INTERACTIVA
# =====================================================================

# Cambiar a TRUE para probar modo interactivo (se hace para que no limite la ejecución del Notebook entero sin bloquearse)
MODO_INTERACTIVO = False

if MODO_INTERACTIVO:
    print("👇 BÚSQUEDA INTERACTIVA 👇")
    query_usuario = input("🔎 ¿Qué consulta quieres realizar?: ")
    if query_usuario:
        n_usuario = input("📊 ¿Qué TOP de noticias quieres ver? (Máx 10): ")
        if n_usuario.isdigit():
            resultados_usuario = consultar_noticias(query_usuario, n_usuario)
            mostrar_resultados(query_usuario, resultados_usuario)
        else:
            print("⚠️ Número inválido. Búsqueda cancelada.")
    else:
        print("⏭️ Búsqueda interactiva omitida.")
else:
    print("ℹ️ Modo interactivo desactivado. Cambia MODO_INTERACTIVO = True al inicio de celda para probarlo manualmente.")


ℹ️ Modo interactivo desactivado. Cambia MODO_INTERACTIVO = True al inicio de celda para probarlo manualmente.


#### 3.2.2   Conclusiones y Valor Aportado




El desarrollo del Motor de Búsqueda Vectorial mediante la arquitectura RAG (*Retrieval-Augmented Generation*) apoyada en FAISS ha transformado un repositorio estático de 6.335 registros en un ecosistema de conocimiento dinámico e interactivo.

* **Integración Tecnológica Avanzada:** A diferencia de los sistemas de búsqueda tradicionales basados en coincidencias literales de palabras clave, la vectorización mediante *embeddings* dota al sistema de comprensión semántica. Esto permite a los administradores de PontiApp rastrear y auditar narrativas complejas de desinformación, conectando conceptos y temáticas afines incluso cuando los autores de las *Fake News* intentan camuflar el contenido usando sinónimos o jerga diferente.
* **Valor Estratégico e Investigación:** Más allá de la moderación en tiempo real, esta herramienta se erige como un pilar fundamental para el análisis de inteligencia dentro de la plataforma. Permite identificar campañas coordinadas de desinformación a lo largo del tiempo, facilitando auditorías históricas profundas de forma casi instantánea y sin requerir conocimientos avanzados de bases de datos por parte del usuario final.

## 4- Conclusiones generales y comparativa de soluciones

El desarrollo de este Proyecto Júpiter demuestra que la integración de LLMs y arquitecturas vectoriales transforma radicalmente la gestión de datos en aplicaciones modernas.

Para **PontiApp**, hemos logrado transicionar de un modelo de "revisión manual exhaustiva" a un modelo de **"Human-in-the-Loop"**. La Inteligencia Artificial asume el trabajo pesado de analizar, puntuar, buscar y justificar grandes volúmenes de texto, mientras que el ser humano mantiene el control final de la toma de decisiones, basándose ahora en información digerida y argumentada.

Desde el punto de vista técnico, se ha garantizado la estabilidad del entorno (evitando conflictos de dependencias mediante `FAISS`), se ha optimizado el consumo de tokens mediante filtros previos de longitud y se ha maximizado la experiencia de usuario (UX) garantizando que todos los *outputs* se ajusten a formatos legibles y estructurados. Este ecosistema híbrido no solo cumple con los requisitos del proyecto, sino que asienta bases sólidas para la escalabilidad futura de la plataforma.

💡 **Futuras Líneas de Evolución:**
Como evolución natural de estos productos para PontiApp, el siguiente hito arquitectónico sería la sustitución de la API comercial actual por un Modelo de Lenguaje Open Source (como Llama 3 o Mistral) ejecutado en infraestructura local o *nubes privadas*. Esta transición eliminaría la dependencia de terceros, garantizaría que los textos sensibles nunca salgan del entorno de la empresa y reforzaría al máximo los estándares de privacidad, seguridad del dato y cumplimiento normativo (RGPD).

**Limitación conocida:** el motor RAG solo puede recuperar información sobre noticias ya indexadas en el momento de construir el índice. Su utilidad para detectar narrativas de desinformación completamente nuevas depende de mantener el índice actualizado (ver "Automatización de Datos" en el apartado de 'Escalabilidad' del documento de IA Generativa).